# 🧪 W6-D6 概念实验：搭建 Function-Calling Agent

> 配套阅读：`第6周-Day6-搭建Function-Calling-Agent.md`（完整循环、教学版/生产版对照在那边）
>
> 今天把 Function Calling 的完整链路拆成可运行的四段：
> 1. **工具定义层**：给 LLM 看的 JSON Schema 说明书 + 参数校验器
> 2. **完整循环**：parse → validate → execute → observation → 循环直到最终回答（含多轮上下文）
> 3. **错误即反馈**：幻觉工具名 / 缺参数 / 类型错误 → 校验错误回传 → 模型自我修复
> 4. **运行数据**：工具调用分布与循环步数的可视化

环境：仅 numpy / 标准库 / matplotlib。`llm_decide()` 模拟模型输出，聚焦协议与循环本身。

## 实验 1：工具定义层 —— Schema 说明书 + 参数校验器

每个工具 = **给人看的说明 + 给 LLM 看的 JSON Schema + 给机器执行的函数** 三位一体。
`validate_call()` 按 Schema 校验模型输出的调用请求：工具名、required、类型、未知参数。

In [ ]:
import json

INVENTORY = {"杨枝甘露": 42, "芒果西米露": 3, "桂花酸梅汤": 0}
ORDERS = {"A1024": {"status": "已发货", "eta": "明天", "paid": 128.0},
          "A2048": {"status": "打包中", "eta": "后天", "paid": 45.0}}

def query_inventory(product: str) -> dict:
    return {"product": product, "stock": INVENTORY.get(product, None)}

def calc_discount(price: float, member: bool) -> dict:
    return {"price": price, "member": member, "final": round(price * (0.8 if member else 1.0), 2)}

TOOL_REGISTRY = {
    "query_inventory": {
        "schema": {"description": "查询糖水店商品库存",
                   "parameters": {"product": {"type": "string", "required": True}}},
        "fn": query_inventory},
    "calc_discount": {
        "schema": {"description": "按会员身份计算折后价",
                   "parameters": {"price": {"type": "number", "required": True},
                                   "member": {"type": "boolean", "required": True}}},
        "fn": calc_discount},
}

def validate_call(call):
    """校验模型输出的 function call，返回错误列表（空=合法）。"""
    errors = []
    name = call.get("name")
    if name not in TOOL_REGISTRY:
        return [f"未知工具 '{name}'，可用工具: {list(TOOL_REGISTRY)}"]
    params = TOOL_REGISTRY[name]["schema"]["parameters"]
    args = call.get("arguments", {})
    for pname, spec in params.items():
        if spec["required"] and pname not in args:
            errors.append(f"缺少必填参数 '{pname}' ({spec['type']})")
    type_map = {"string": str, "number": (int, float), "boolean": bool}
    for pname, val in args.items():
        if pname not in params:
            errors.append(f"未知参数 '{pname}'"); continue
        want = type_map[params[pname]["type"]]
        if not isinstance(val, want) or (want == (int, float) and isinstance(val, bool)):
            errors.append(f"参数 '{pname}' 应为 {params[pname]['type']}，收到 {type(val).__name__}")
    return errors

tests = [
    {"name": "query_inventory", "arguments": {"product": "芒果西米露"}},   # 合法
    {"name": "query_weather",   "arguments": {"city": "上海"}},           # 幻觉工具
    {"name": "calc_discount",   "arguments": {"price": 45}},              # 缺 member
    {"name": "calc_discount",   "arguments": {"price": "45", "member": True}},  # 类型错
]
for t in tests:
    errs = validate_call(t)
    print(f"{'✓' if not errs else '✗'} {t['name']}({t['arguments']})" +
          (f"  → {'; '.join(errs)}" if errs else "  → 放行执行"))
print("\n校验器是防线1：模型的输出永远不可信，先验后执行。")

## 实验 2：完整循环 + 多轮上下文

主循环：`llm_decide()` 模拟模型在每轮看到 **系统提示 + 对话历史（含工具观测）** 后输出
function_call 或最终回答。跑两个场景：单工具查询、需要两工具串联+多轮上下文的复合请求。

In [ ]:
def llm_decide(messages):
    """模拟 LLM：真实系统里这里调 API 并解析返回的 tool_calls。"""
    user_q = next((m["content"] for m in messages if m["role"] == "user"), "")
    called = {m["call"]["name"] for m in messages if m["role"] == "tool_call"}
    if ("订单" in user_q or "A1024" in user_q) and "query_order" not in called:
        return {"call": {"name": "query_order", "arguments": {"order_id": "A1024"}}}
    if ("折扣" in user_q or "会员" in user_q or "便宜" in user_q) and "calc_discount" not in called:
        price = ORDERS["A1024"]["paid"]        # 从上一轮工具观测里取实付金额（工具链串联）
        return {"call": {"name": "calc_discount", "arguments": {"price": price, "member": True}}}
    # 工具都调用过了 → 基于历史里的观测总结回答
    obs = [m for m in messages if m["role"] == "tool_result"]
    return {"final": f"已结合 {len(obs)} 条工具结果生成回答"}

TOOL_REGISTRY["query_order"] = {
    "schema": {"description": "查询订单状态",
               "parameters": {"order_id": {"type": "string", "required": True}}},
    "fn": lambda order_id: ORDERS[order_id]}

def run_agent(question, max_turns=6):
    messages = [{"role": "user", "content": question}]
    print(f"用户: {question}")
    for turn in range(1, max_turns + 1):
        out = llm_decide(messages)
        if "final" in out:
            print(f"[turn{turn}] Agent 最终回答: {out['final']} ✓\n")
            return messages
        call = out["call"]
        errs = validate_call(call)
        if errs:
            messages.append({"role": "tool_call", "call": call})
            messages.append({"role": "tool_error", "content": "; ".join(errs)})
            print(f"[turn{turn}] 校验失败: {errs}（下一轮把错误喂回去）")
            continue
        result = TOOL_REGISTRY[call["name"]]["fn"](**call["arguments"])
        messages += [{"role": "tool_call", "call": call},
                     {"role": "tool_result", "content": json.dumps(result, ensure_ascii=False)}]
        print(f"[turn{turn}] 调用 {call['name']}({call['arguments']}) → 观测: {result}")
    print("[abort] 超过最大轮数")

run_agent("订单 A1024 现在什么状态？")
run_agent("我是会员，按 A1024 的实付金额算，会员价能便宜到多少？")

## 实验 3：错误即反馈 —— 自愈循环

模型会犯：幻觉工具、缺参、类型错。关键是**把校验错误作为 observation 喂回下一轮**，
而不是直接崩。模拟"第一轮必错、看到错误信息后修正"的自愈过程，并统计 2000 次任务里
'1 轮修复 / 2 轮修复 / 彻底失败'的分布（修复概率 p=0.85/轮）。

In [ ]:
import numpy as np
rng = np.random.default_rng(21)

BAD_CALLS = [
    {"name": "query_stock",   "arguments": {"product": "杨枝甘露"}},   # 幻觉工具名
    {"name": "calc_discount", "arguments": {"price": 45}},             # 缺参
]
def model_with_feedback(messages):
    """模拟：首轮输出坏调用；看到 tool_error 后以 85% 概率修正。"""
    if not any(m["role"] == "tool_error" for m in messages):
        return {"call": dict(rng.choice(BAD_CALLS))}
    if rng.random() < 0.85:   # 读懂错误信息 → 修正
        return {"call": {"name": "calc_discount", "arguments": {"price": 45, "member": True}}}
    return {"call": dict(rng.choice(BAD_CALLS))}   # 仍没修对

def self_heal_run(max_turns=4):
    messages, bad = [], 0
    for _ in range(max_turns):
        out = model_with_feedback(messages)
        call = out["call"]
        errs = validate_call(call)
        if not errs:
            return "ok_first_try" if bad == 0 else f"fixed_after_{bad}_errors", messages
        bad += 1
        messages.append({"role": "tool_error", "content": "; ".join(errs)})
    return "failed", messages

outcome, msgs = self_heal_run()
print("样例运行轨迹:")
for m in msgs:
    print(f"  [{m['role']}] {m['content'][:70]}")
print(f"  → 结果: {outcome}\n")

N = 2000
dist = {}
for _ in range(N):
    o, _ = self_heal_run()
    dist[o] = dist.get(o, 0) + 1
for k in sorted(dist):
    print(f"{k:<22}{dist[k]/N:>7.1%}")
print("\n解读：把'错误描述'写清楚（要什么类型、哪些工具可用），修复率会显著上升——")
print("错误信息本身就是 prompt 的一部分。")

## 实验 4：运行数据可视化 —— 调用分布与循环步数

In [ ]:
# matplotlib 中文字体配置（NotoSansCJK，每次画图前先跑这段）
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = fontManager_font = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

import numpy as np
rng = np.random.default_rng(33)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))

# 左：模拟 200 次运行的工具调用分布
tools = ["query_inventory", "query_order", "calc_discount"]
weights = [0.45, 0.35, 0.20]
calls = rng.choice(tools, size=800, p=weights)
counts = {t: (calls == t).sum() for t in tools}
axes[0].bar(counts.keys(), counts.values(), color=["#219ebc", "#fb8500", "#2a9d8f"])
for i, v in enumerate(counts.values()):
    axes[0].text(i, v + 6, str(v), ha="center", fontsize=10)
axes[0].set_ylabel("调用次数（800 次工具调用）")
axes[0].set_title("工具调用分布（监控：谁被滥用/谁没人用）")
axes[0].grid(alpha=0.3, axis="y")

# 右：每次任务循环步数分布（多工具任务会拖长循环）
steps_1tool = rng.poisson(2.0, 200)      # 单工具任务 ≈ 2-3 轮
steps_multi = rng.poisson(4.5, 200)      # 多工具串联 ≈ 4-6 轮
axes[1].hist([steps_1tool, steps_multi], bins=range(0, 11),
             label=["单工具任务", "多工具任务"], color=["#adb5bd", "#fb8500"])
axes[1].set_xlabel("Agent 循环轮数"); axes[1].set_ylabel("任务数")
axes[1].set_title("循环步数分布（max_turns 要罩住右尾）")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print("生产提示：这两个图就是 Function Calling Agent 的第一批监控面板——")
print("调用分布看成本与滥用，步数分布看 max_turns 与延迟预算是否合理。")

## 小结

- 工具 = 人读描述 + LLM 读 Schema + 机器执行函数；`validate_call` 先验后执行
- 完整循环：parse → validate → execute → observation → 回合，多轮靠 messages 累积
- 错误信息是 prompt 的一部分：描述清楚，模型能自愈（85%+ 一轮修复）
- 教学版→生产版只换 `llm_decide()` 一个函数：协议不变，这就是分层的价值